# Met Eyes Experiments

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

In [ ]:
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils.py

In [ ]:
import json
import requests

from os import listdir, makedirs, path
from PIL import Image as PImage
from time import sleep

from utils import export_combined_jsons

DATA_DIR = "./data"
IMG_DIR = f"{DATA_DIR}/image"
JSON_DIR = f"{DATA_DIR}/json"
OBJS_DIR = f"{JSON_DIR}/objects"

makedirs(DATA_DIR, exist_ok=True)
makedirs(IMG_DIR, exist_ok=True)
makedirs(JSON_DIR, exist_ok=True)
makedirs(OBJS_DIR, exist_ok=True)

In [ ]:
MET_URL = "https://collectionapi.metmuseum.org/public/collection/v1"

SEARCH_DEPARTMENT_IDS = []
SEARCH_DEPARTMENTS = ["Robert Lehman", "Armor"][:1]
SEARCH_MEDIUMS = ["Paintings", "Drawings"][:1]

### Get Department IDs

In [ ]:
dept_response = requests.get(f"{MET_URL}/departments")
dept_data = dept_response.json()["departments"]

dept_name2id = { d["displayName"] : d["departmentId"] for d in dept_data }

for sdpt in SEARCH_DEPARTMENTS:
  for dname,did in dept_name2id.items():
    if sdpt.lower() in dname.lower():
      SEARCH_DEPARTMENT_IDS.append(did)

### Get Object IDs

In [ ]:
obj_ids = []

for dpt_query in SEARCH_DEPARTMENT_IDS:
  for medium_query in SEARCH_MEDIUMS:
    collection_response = requests.get(f"{MET_URL}/search?medium={medium_query}&departmentId={dpt_query}&q=*")
    query_obj_ids = set(collection_response.json()["objectIDs"])
    obj_ids += list(query_obj_ids)

len(obj_ids)

### Get Object Metadata

In [ ]:
obj_fields = ["objectID", "objectName", "title", "primaryImage", "primaryImageSmall", "artistRole", "artistDisplayName"]
obj_files = sorted(f for f in listdir(OBJS_DIR) if f.endswith("json"))

for cnt,oid in enumerate(obj_ids):
  if cnt % 20 == 0:
    print(f"{cnt} / {len(obj_ids)}")

  obj_json_path = f"{OBJS_DIR}/{oid}.json"
  if f"{oid}.json" in obj_files:
    continue

  obj_response = requests.get(f"{MET_URL}/objects/{oid}")
  obj_data = obj_response.json()
  obj_filtered_data = { f: obj_data[f] for f in obj_fields }

  obj_json_path = f"{OBJS_DIR}/{oid}.json"
  with open(obj_json_path, "w") as ofp:
    json.dump(obj_filtered_data, ofp)
  sleep(0.333)

### Export Combined Object Metadata

In [ ]:
export_combined_jsons(f"{JSON_DIR}/objects", JSON_DIR, "objects")

### Get Images

In [ ]:
for size in ["500", "900"]:
  makedirs(f"{IMG_DIR}/{size}/", exist_ok=True)

with open(f"{JSON_DIR}/objects.json", "r") as ifp:
  obj_data = json.load(ifp)["objects"]

for cnt,obj in enumerate(obj_data):
  if cnt % 20 == 0:
    print(f"{cnt} / {len(obj_data)}")

  img_url = obj["primaryImage"]
  if not (img_url and len(img_url) > 0):
    continue

  oid = obj["objectID"]

  img_900_path = f"{IMG_DIR}/900/{oid}.jpg"
  img_500_path = f"{IMG_DIR}/500/{oid}.jpg"
  if path.isfile(img_900_path) and path.isfile(img_500_path):
    continue

  img_response = requests.get(img_url, stream=True)
  img = PImage.open(img_response.raw)

  if not path.isfile(img_900_path):
    img.thumbnail((900, 900))
    img.save(img_900_path)

  if not path.isfile(img_500_path):
    img.thumbnail((500, 500))
    img.save(img_500_path)

  sleep(0.333)

## Analyze Faces

### Face Detection

In [ ]:
!git clone https://PAT_XYZ@github.com/acervos-digitais/met-faces-data.git data
!pip install ultralytics

In [ ]:
import json
import numpy as np

from os import listdir
from PIL import Image as PImage, ImageDraw as PImageDraw

from huggingface_hub import hf_hub_download
from torch import no_grad, Tensor
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, pipeline
from ultralytics import YOLO

from utils import export_combined_jsons, pxs_to_pcts, draw_boxes, pcts_to_sqs

DATA_DIR = "./data"
IMG_DIR = f"{DATA_DIR}/image"
JSON_DIR = f"{DATA_DIR}/json"

In [ ]:
img_ids = sorted(int(fn.replace(".jpg", "")) for fn in listdir(f"{IMG_DIR}/900") if fn.endswith(".jpg"))

with open(f"{JSON_DIR}/objects.json", "r") as ifp:
  obj_data = json.load(ifp)["objects"]
  id2obj = { obj["objectID"] : obj for obj in obj_data }

### Detect Faces ([YOLO11](https://huggingface.co/AdamCodd/YOLOv11n-face-detection))

In [ ]:
yolo_model_path = hf_hub_download(repo_id="AdamCodd/YOLOv11n-face-detection", filename="model.pt")
face_detector = YOLO(yolo_model_path)

In [ ]:
for cnt,oid in enumerate(img_ids):
  if cnt % 20 == 0:
    print(f"{cnt} / {len(img_ids)}")

  obj = id2obj[oid]

  if "faces" in obj and "yolo" in obj["faces"]:
    continue

  img = PImage.open(f"{IMG_DIR}/900/{oid}.jpg")
  iw,ih = img.size
  nh = 256
  nw = int(nh * iw // ih)
  nimg = img.resize((nw, nh))

  faces = face_detector.predict(nimg, verbose=False, device="cuda")
  if len(faces) < 1 or len(faces[0]) < 1:
    continue

  faces_xyxyn = faces[0].boxes.xyxyn.cpu().numpy().astype(np.float64)
  faces_xyxyn_sq = pcts_to_sqs(faces_xyxyn, iw, ih)
  # faces_xywhn = faces[0].boxes.xywhn.cpu().numpy().astype(np.float64)

  if "faces" not in obj:
    obj["faces"] = {}

  obj["faces"]["yolo"] = {
      "count": len(faces_xyxyn),
      "xyxyn": faces_xyxyn.round(4).tolist(),
      "xyxyn_sq": faces_xyxyn_sq.round(4).tolist(),
      # "xywhn": faces_xywhn.round(4).tolist(),
  }

  with open(f"{JSON_DIR}/objects/{oid}.json", "w") as ofp:
    json.dump(obj, ofp)

### Detect Faces (Zero-Shot)

In [ ]:
MODEL_NAME = "IDEA-Research/grounding-dino-base"

zs_processor = AutoProcessor.from_pretrained(MODEL_NAME)
zs_model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_NAME).to("cuda")

labels = ["face"]

In [ ]:
for cnt,oid in enumerate(img_ids):
  if cnt % 20 == 0:
    print(f"{cnt} / {len(img_ids)}")

  obj = id2obj[oid]

  if "faces" in obj and "dino" in obj["faces"]:
    continue

  img = PImage.open(f"{IMG_DIR}/900/{oid}.jpg")
  iw,ih = img.size

  with no_grad():
    input = zs_processor(text=labels, images=img, return_tensors="pt").to("cuda")
    output = zs_model(**input)

  res = zs_processor.post_process_grounded_object_detection(outputs=output, target_sizes=[Tensor([ih, iw])], threshold=0.33)

  if len(res[0]["boxes"]) < 1:
    continue

  faces_xyxyn = pxs_to_pcts(res[0]["boxes"].cpu(), iw, ih, xyxy=True).astype(np.float64)
  faces_xyxyn_sq = pcts_to_sqs(faces_xyxyn, iw, ih)
  # faces_xywhn = pxs_to_pcts(res[0]["boxes"].cpu(), iw, ih, xyxy=False).astype(np.float64)

  if "faces" not in obj:
    obj["faces"] = {}

  obj["faces"]["dino"] = {
      "count": len(res[0]["boxes"]),
      "xyxyn": faces_xyxyn.round(4).tolist(),
      "xyxyn_sq": faces_xyxyn_sq.round(4).tolist(),
      # "xywhn": faces_xywhn.round(4).tolist(),
  }

  with open(f"{JSON_DIR}/objects/{oid}.json", "w") as ofp:
    json.dump(obj, ofp)

In [ ]:
export_combined_jsons(f"{JSON_DIR}/objects", JSON_DIR, "faces", ["faces"])

### Crop Faces

In [ ]:
import json
import numpy as np
import requests

from os import listdir, makedirs
from PIL import Image as PImage, ImageDraw as PImageDraw
from time import sleep

from utils import pct_to_px

DATA_DIR = "./data"
IMG_DIR = f"{DATA_DIR}/image"
FACE_DIR = f"{IMG_DIR}/faces"
JSON_DIR = f"{DATA_DIR}/json"

makedirs(FACE_DIR, exist_ok=True)

In [ ]:
face_img_ids = set(int(f.split("_")[0]) for f in listdir(FACE_DIR) if f.endswith("jpg"))

print(len(face_img_ids))

with open(f"{JSON_DIR}/faces.json", "r") as ifp:
  obj_data = json.load(ifp)["faces"]
  id2obj = { obj["objectID"] : obj for obj in obj_data }

In [ ]:
for ocnt,obj in enumerate(obj_data):
  if ocnt % 20 == 0:
    print(f"{ocnt} / {len(obj_data)}")

  oid = obj["objectID"]

  if oid in face_img_ids:
    continue

  if not ("faces" in obj and "yolo" in obj["faces"]):
    continue

  img_response = requests.get(obj["primaryImage"], stream=True)
  img = PImage.open(img_response.raw)
  iw,ih = img.size

  for fcnt,box in enumerate(obj["faces"]["yolo"]["xyxyn_sq"]):
    x0,y0,x1,y1 = pct_to_px(box, iw, ih)
    bw, bh = (x1 - x0), (y1 - y0)

    if (bw > 0.9 * iw) or (bh > 0.9 * ih) or (bw < 100) or (bh < 100):
      continue

    face_img_cnt_str = f"000{fcnt}"[-3:]
    face_img_fname = f"{oid}_{face_img_cnt_str}"
    face_img = img.crop((x0,y0,x1,y1)).convert("RGB")
    face_img.save(f"{IMG_DIR}/faces/{face_img_fname}.jpg")

  face_img_ids.add(oid)
  sleep(0.333)

### Face Landmarks: Eyes and Gaze

In [ ]:
# TODO: Landmark Detection
  # TODO: OpenCV
  # TODO: Dino
  # TODO: https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker/index
  # TODO: https://huggingface.co/kartiknarayan/facexformer
  # TODO: https://huggingface.co/qualcomm/Facial-Landmark-Detection